# Lab 05 - Panoramas com diferentes descritores

Para cada descritor analisado no artigo *A Comparative Analysis of SIFT, SURF, KAZE, AKAZE, ORB and BRISK*,
montamos uma imagem panoramica a partir das duas fotos capturadas (`centro.jpeg` -> vista esquerda e `direita.jpeg` -> vista direita).

O pipeline segue a mesma ideia dos arquivos de referencia:
- **`descriptors.py`** -> `detectAndCompute` para extrair keypoints e descritores.
- **`matching.py`** -> casamento com `BFMatcher` + *ratio test* de Lowe.
- A homografia (`RANSAC`) alinha as imagens e o `warpPerspective` as costura.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print('OpenCV:', cv2.__version__)

## 1. Carregando as duas imagens

`centro.jpeg` cobre a parte esquerda da cena e `direita.jpeg` a parte direita; elas se sobrepoem na regiao do monitor.

In [ ]:
img_esquerda = cv2.imread('centro.jpeg')   # vista esquerda
img_direita  = cv2.imread('direita.jpeg')  # vista direita

assert img_esquerda is not None and img_direita is not None, 'Falha ao ler as imagens'
print('esquerda:', img_esquerda.shape, '| direita:', img_direita.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(cv2.cvtColor(img_esquerda, cv2.COLOR_BGR2RGB)); ax[0].set_title('centro.jpeg (esquerda)'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(img_direita,  cv2.COLOR_BGR2RGB)); ax[1].set_title('direita.jpeg (direita)'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 2. Funcoes auxiliares

- `detect_and_describe`: extrai keypoints/descritores (como em `descriptors.py`).
- `match_keypoints`: casa descritores com `BFMatcher` e aplica o *ratio test* (como em `matching.py`).
- `stitch`: estima a homografia com RANSAC e costura as imagens em uma panoramica.

Descritores **binarios** (ORB, AKAZE, BRISK) usam distancia de Hamming; descritores **de ponto flutuante** (SIFT, SURF, KAZE) usam distancia L2.

In [ ]:
def detect_and_describe(detector, image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    return keypoints, descriptors


def match_keypoints(des1, des2, norm, ratio=0.75):
    """BFMatcher com knnMatch + ratio test de Lowe (ver matching.py)."""
    bf = cv2.BFMatcher(norm)
    raw_matches = bf.knnMatch(des1, des2, k=2)
    good = []
    for pair in raw_matches:
        if len(pair) == 2:
            m, n = pair
            if m.distance < ratio * n.distance:
                good.append(m)
    return good


def stitch(img_left, img_right, kp1, kp2, matches):
    """Alinha a imagem da direita ao plano da esquerda e costura as duas."""
    if len(matches) < 4:
        raise ValueError('Casamentos insuficientes para estimar a homografia (>= 4).')
    pts_left  = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts_right = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(pts_right, pts_left, cv2.RANSAC, 4.0)

    h1, w1 = img_left.shape[:2]
    h2, w2 = img_right.shape[:2]
    panorama = cv2.warpPerspective(img_right, H, (w1 + w2, max(h1, h2)))
    panorama[0:h1, 0:w1] = img_left

    # recorta a faixa preta a direita (colunas totalmente vazias)
    gray = cv2.cvtColor(panorama, cv2.COLOR_BGR2GRAY)
    cols = np.where(gray.sum(axis=0) > 0)[0]
    if len(cols) > 0:
        panorama = panorama[:, :cols.max() + 1]
    return panorama

## 3. Descritores do artigo

Definimos cada detector/descritor com a norma de distancia adequada. O **SURF** e patenteado (modulo `opencv-contrib` *non-free*); caso nao esteja disponivel no build instalado, ele e ignorado com aviso.

In [ ]:
detectores = {}
detectores['SIFT'] = (cv2.SIFT_create(), cv2.NORM_L2)

try:
    detectores['SURF'] = (cv2.xfeatures2d.SURF_create(), cv2.NORM_L2)
except Exception as e:
    print('SURF indisponivel neste build do OpenCV (patenteado/non-free):', e)

detectores['KAZE']  = (cv2.KAZE_create(),  cv2.NORM_L2)
detectores['AKAZE'] = (cv2.AKAZE_create(), cv2.NORM_HAMMING)
detectores['ORB']   = (cv2.ORB_create(nfeatures=2000), cv2.NORM_HAMMING)
detectores['BRISK'] = (cv2.BRISK_create(), cv2.NORM_HAMMING)

print('Descritores a testar:', list(detectores.keys()))

## 4. Panoramica para cada descritor

In [ ]:
panoramas = {}

for nome, (detector, norm) in detectores.items():
    print('=' * 60)
    print(f'Descritor: {nome}')
    try:
        kp1, des1 = detect_and_describe(detector, img_esquerda)
        kp2, des2 = detect_and_describe(detector, img_direita)
        print(f'  Keypoints  -> esquerda: {len(kp1)} | direita: {len(kp2)}')

        matches = match_keypoints(des1, des2, norm)
        print(f'  Bons casamentos (ratio test): {len(matches)}')

        panorama = stitch(img_esquerda, img_direita, kp1, kp2, matches)
        panoramas[nome] = panorama
        print(f'  Panoramica gerada: {panorama.shape}')
    except Exception as e:
        print(f'  [ERRO] Nao foi possivel gerar a panoramica com {nome}: {e}')

## 5. Resultados

In [ ]:
n = len(panoramas)
fig, axes = plt.subplots(n, 1, figsize=(14, 6 * n))
if n == 1:
    axes = [axes]

for ax, (nome, pano) in zip(axes, panoramas.items()):
    ax.imshow(cv2.cvtColor(pano, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Panoramica - {nome}', fontsize=14)
    ax.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
# Salva cada panoramica em disco
for nome, pano in panoramas.items():
    saida = f'panorama_{nome}.jpg'
    cv2.imwrite(saida, pano)
    print('Salvo:', saida)